# Telecom Customer Churn Analysis
## 02 — Data Cleaning


## Cleaning Objective
The data audit identified several missing-value patterns and potential data-quality issues.

A separate working copy of the raw dataset will be created so that the original source data remains unchanged.

Cleaning decisions will be based on the findings from the initial audit.

The goal of this stage was to address the missing-value patterns and data-quality issues identified during the data audit while avoiding unsupported assumptions about the source data.


In [1]:
import pandas as pd

In [31]:
# Load the original dataset 

dataset = r"C:\Users\HomePC\Desktop\Data Analysis Portfolio\telecom_customer_churn_analysis\data\telecom_customer_churn.csv"
df = pd.read_csv(dataset)

In [32]:
# Create a copy of the original dataset

df_clean = df.copy()

In [34]:
# Confirm the original data is untouched

df_clean.shape

(7043, 38)

### Cleaning internet-related fields 

In [36]:
# Check the actual values of the internet-related fields with missing values

internet_cols =[
    "Internet Type",
    "Avg Monthly GB Download",
    "Online Security",
    "Online Backup",
    "Device Protection Plan",
    "Premium Tech Support",
    "Streaming TV",
    "Streaming Movies",
    "Streaming Music",
    "Unlimited Data"
]

df_clean.loc[
    df_clean["Internet Service"] == "No",
    internet_cols
].head(10)

,Internet Type,Avg Monthly GB Download,Online Security,Online Backup,Device Protection Plan,Premium Tech Support,Streaming TV,Streaming Movies,Streaming Music,Unlimited Data
20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
38,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
# Clean the internet-related fields

no_internet = df_clean["Internet Service"] == "No"

df_clean.loc[no_internet, "Internet Type"] = "No Internet"
df_clean.loc[no_internet, "Avg Monthly GB Download"] = 0

internet_feature_cols = [
    "Online Security",
    "Online Backup",
    "Device Protection Plan",
    "Premium Tech Support",
    "Streaming TV",
    "Streaming Movies",
    "Streaming Music",
    "Unlimited Data"
]

df_clean.loc[no_internet, internet_feature_cols] = "No"

In [7]:
df_clean[internet_cols].isna().sum()

Internet Type              0
Avg Monthly GB Download    0
Online Security            0
Online Backup              0
Device Protection Plan     0
Premium Tech Support       0
Streaming TV               0
Streaming Movies           0
Streaming Music            0
Unlimited Data             0
dtype: int64

In [8]:
df_clean.loc[
    df_clean["Internet Service"] == "No",
    internet_cols
].head()

,Internet Type,Avg Monthly GB Download,Online Security,Online Backup,Device Protection Plan,Premium Tech Support,Streaming TV,Streaming Movies,Streaming Music,Unlimited Data
20,No Internet,0.0,No,No,No,No,No,No,No,No
23,No Internet,0.0,No,No,No,No,No,No,No,No
24,No Internet,0.0,No,No,No,No,No,No,No,No
27,No Internet,0.0,No,No,No,No,No,No,No,No
28,No Internet,0.0,No,No,No,No,No,No,No,No


### Cleaning Phone-Service-related fields

In [9]:
no_phone = df_clean["Phone Service"] == "No"

df_clean.loc[no_phone, "Multiple Lines"] = "No"

df_clean["Multiple Lines"].isna().sum()

np.int64(0)

In [10]:
pd.crosstab(
    df_clean["Phone Service"],
    df_clean["Multiple Lines"],
    dropna=False
)

Multiple Lines,No,Yes
Phone Service,,
No,682,0
Yes,3390,2971


In [44]:
# Missing values for 'Avg Monthly Long Distance Charges' correspond exactly to customers with no phone service. 
# The data dictionary says the value should be 0 when there is no phone service. So we'll clean it similarly.


df_clean.loc[no_phone, "Avg Monthly Long Distance Charges"] = 0

In [42]:
df_clean["Avg Monthly Long Distance Charges"].isna().sum()

np.int64(0)

In [43]:
df_clean.loc[
    df_clean["Phone Service"] == "No",
    ["Phone Service", "Avg Monthly Long Distance Charges", "Multiple Lines"]
].head()

,Phone Service,Avg Monthly Long Distance Charges,Multiple Lines
10,No,0.0,NaN
14,No,0.0,NaN
16,No,0.0,NaN
19,No,0.0,NaN
25,No,0.0,NaN


### Cleaning Customer churn fields

In [11]:
not_churned = df_clean["Customer Status"] != "Churned"

df_clean.loc[not_churned, "Churn Category"] = "Not Churned"
df_clean.loc[not_churned, "Churn Reason"] = "Not Churned"

In [12]:
df_clean[["Churn Category", "Churn Reason"]].isna().sum()

Churn Category    0
Churn Reason      0
dtype: int64

In [45]:
# Validate

pd.crosstab(
    df_clean["Customer Status"],
    df_clean["Churn Category"],
    dropna=False
)

Churn Category,Attitude,Competitor,Dissatisfaction,Other,Price,NaN
Customer Status,,,,,,
Churned,314,841,321,182,211,0
Joined,0,0,0,0,0,454
Stayed,0,0,0,0,0,4720


### Investigate Offer column

In [39]:
df_clean["Offer"].unique()

array([nan, 'Offer E', 'Offer D', 'Offer A', 'Offer B', 'Offer C'],
      dtype=object)

In [15]:
df_clean["Offer"].isna().sum()

np.int64(3877)

In [40]:
# More than half of the customers have 3,877 missing Offer values
# I can't established that those customers should have a particular offer value. Or that the value should be 'None'.
# First, inspect whether the missing values correspond to anything obvious in the source data.

df_clean.groupby(
    df_clean["Offer"].isna()
)["Tenure in Months"].describe()

,count,mean,std,min,25%,50%,75%,max
Offer,,,,,,,,
False,3166.0,33.391661,25.095938,1.0,9.0,29.0,58.0,72.0
True,3877.0,31.566159,24.052436,1.0,8.0,29.0,53.0,72.0


In [17]:
pd.crosstab(
    df_clean["Offer"].isna(),
    df_clean["Customer Status"],
    normalize="index"
).round(3)

Customer Status,Churned,Joined,Stayed
Offer,,,
False,0.258,0.055,0.686
True,0.271,0.072,0.657


### Offer Cleaning Decision

The `Offer` field contains 3,877 missing values.

The data dictionary defines `Offer` as the last marketing offer accepted and identifies `None` as a valid value meaning that no offer was accepted. However, the dataset contains no explicit `None` category; the corresponding records are stored as missing values (`NaN`).

Further investigation showed that customers with missing Offer values have similar tenure and customer-status distributions to customers with recorded offers. The available evidence does not establish whether the missing values specifically represent customers who did not accept an offer.

Therefore, the missing `Offer` values will be preserved as `NaN` rather than being converted to `None` or another category without sufficient evidence.

### Investigate negative Monthly Charge

In [46]:
# Investigate whether the negative values conflict with the customer's other financial information.

negative_monthly = df_clean["Monthly Charge"] < 0

df_clean.loc[
    negative_monthly,
    [
        "Customer ID",
        "Monthly Charge",
        "Tenure in Months",
        "Total Charges",
        "Total Refunds",
        "Total Extra Data Charges",
        "Total Long Distance Charges",
        "Total Revenue",
        "Customer Status"
    ]
].head(10)

,Customer ID,Monthly Charge,Tenure in Months,Total Charges,Total Refunds,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Customer Status
1,0003-MKNFE,-4.0,9,542.40,38.33,10,96.21,610.28,Stayed
32,0052-DCKON,-2.0,66,7942.15,0.00,0,2888.82,10830.97,Stayed
170,0274-JKUJR,-3.0,7,465.70,0.00,0,0.00,465.70,Stayed
232,0361-HJRDX,-8.0,68,4539.60,0.00,130,0.00,4669.60,Stayed
336,0495-ZBNGW,-1.0,5,343.95,0.00,20,33.30,397.25,Churned
428,0623-IIHUG,-10.0,21,840.10,0.00,0,0.00,840.10,Churned
571,0840-DCNZE,-2.0,47,4138.90,0.00,0,1728.66,5867.56,Stayed
692,1024-GUALD,-4.0,1,24.80,0.00,0,0.00,24.80,Churned
694,1024-VRZHF,-7.0,11,825.70,0.00,0,237.38,1063.08,Churned
702,1036-GUDCL,-9.0,60,4663.40,0.00,0,2385.60,7049.00,Stayed


In [22]:
df_clean.loc[
    negative_monthly,
    ["Monthly Charge", "Total Charges", "Total Revenue"]
].describe()

,Monthly Charge,Total Charges,Total Revenue
count,120.000000,120.000000,120.000000
mean,-5.416667,2028.856250,2727.278417
std,2.888934,2120.436069,2719.182437
min,-10.000000,19.400000,22.540000
25%,-8.000000,343.025000,460.800000
50%,-5.000000,1325.975000,1920.660000
75%,-3.000000,3239.512500,4410.295000
max,-1.000000,7942.150000,10830.970000


### Monthly Charge Cleaning Decision

The `Monthly Charge` field contains 120 negative values ranging from -1 to -10.

These values are inconsistent with the other financial fields. Customers with negative monthly charges still have positive Total Charges and Total Revenue, indicating that the negative values do not represent an overall negative financial balance.

The underlying cause of these negative values cannot be established from the available data. Therefore, the values will not be converted to positive numbers, since doing so would introduce an unsupported assumption.

Instead, negative `Monthly Charge` values will be treated as invalid observations and replaced with missing values (`NaN`). This preserves the uncertainty and prevents the anomalous values from affecting subsequent analysis of monthly charges.

In [23]:
df_clean.loc[
    df_clean["Monthly Charge"] < 0,
    "Monthly Charge"
] = pd.NA

In [24]:
df_clean["Monthly Charge"].isna().sum()

np.int64(120)

In [25]:
df_clean["Monthly Charge"].describe()

count    6923.000000
mean       64.792366
std        30.107716
min        18.250000
25%        35.500000
50%        70.350000
75%        89.900000
max       118.750000
Name: Monthly Charge, dtype: float64

In [26]:
print("Shape:", df_clean.shape)

print("\nMissing values:")
print(df_clean.isna().sum()[df_clean.isna().sum() > 0])

print("\nDuplicate rows:", df_clean.duplicated().sum())

print("Duplicate Customer IDs:", df_clean["Customer ID"].duplicated().sum())

Shape: (7043, 38)

Missing values:
Offer             3877
Monthly Charge     120
dtype: int64

Duplicate rows: 0
Duplicate Customer IDs: 0


# Data Cleaning Summary

## Cleaning Performed

### 1. Internet Service Fields

Customers without internet service had missing values across several internet-related fields.

These values were handled according to the structure documented in the data dictionary:

* `Internet Type` → `"No Internet"`
* `Avg Monthly GB Download` → `0`
* Internet service features → `"No"`

### 2. Phone Service Fields

Customers without phone service had missing values for `Multiple Lines` and `Avg Monthly Long Distance Charges`.

These were set to:

* `Multiple Lines` → `"No"`
* `Avg Monthly Long Distance Charges` → `0`

### 3. Churn Fields

`Churn Category` and `Churn Reason` were missing for customers who had not churned.

These were replaced with:

* `Churn Category` → `"Not Churned"`
* `Churn Reason` → `"Not Churned"`

This makes the fields easier to interpret during analysis while preserving the distinction between churned and non-churned customers.

### 4. Offer Field

The `Offer` field contains 3,877 missing values.

Although the data dictionary identifies `"None"` as a possible offer value, the dataset stores these records as missing values rather than an explicit `"None"` category.

Investigation of tenure and customer-status distributions did not provide sufficient evidence to determine that all missing values represent customers who did not accept an offer.

Therefore, the missing values were preserved as `NaN`.

### 5. Monthly Charge

The dataset contained 120 negative `Monthly Charge` values ranging from -1 to -10.

These values were inconsistent with the positive `Total Charges` and `Total Revenue` recorded for the same customers. However, the correct values could not be determined from the available data.

The negative values were therefore treated as invalid observations and replaced with `NaN` rather than being converted to positive values.

## Final Validation

After cleaning:

* Rows: **7,043**
* Columns: **38**
* Duplicate rows: **0**
* Duplicate Customer IDs: **0**
* Missing `Offer` values: **3,877**
* Missing `Monthly Charge` values: **120**

No rows were removed during cleaning.

## Cleaning Principle

The cleaning process prioritized preserving the original information and avoiding unsupported assumptions. Values were changed only where their meaning could be reasonably established from the data structure or where clearly invalid values required isolation from subsequent analysis.



In [27]:
df_clean.to_csv(
    "../data/telecom_customer_churn_clean.csv",
    index=False
)